[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hudson11111/MMA3001Project/blob/main/notebooks/04_test_performance_robustness.ipynb)

# 04 — One-shot TEST, speed/optimisation study, robustness, baseline features

**Runtime: T4 GPU** · about 30–40 min · the last GPU session of the project.

| Section | Work package | What it produces |
|---|---|---|
| 4 | WP7 | **One-shot TEST** predictions of the selected model (D-025). Decision thresholds were frozen on VALID *before* this notebook existed (`configs/decision.yaml`, D-026). |
| 5 | WP6 | Accuracy vs **input resolution** and **FP16 vs FP32** (VALID predictions) |
| 6 | WP6 | **Latency** vs batch size, resolution, precision, GPU vs CPU; GPU memory; profiler breakdown |
| 7 | WP6 | **FLOPs and arithmetic intensity** (own counter, checked against Ultralytics) → roofline |
| 8 | WP7 | **Robustness**: exposure, blur, noise, JPEG on VALID |
| 9 | WP4 | Hand-crafted **features for the scikit-learn baseline** |

Scoring happens afterwards on CPU from the saved files, with the tested `porkqc.metrics`.
**Safe to re-run:** TEST predictions are written once to Drive and never overwritten.

## 1. Settings

In [ ]:
RUN_NAME = "yolo11n_640_blocked"   # selected model (D-025)
QUICK = False                      # True only for a fast smoke test (fewer repeats/levels)
PUSH_RESULTS = True

## 2. Code, environment, Drive

In [ ]:
import os, sys, json, subprocess, datetime, platform, shutil
REPO_URL = "https://github.com/Hudson11111/MMA3001Project.git"
REPO_DIR = "/content/MMA3001Project"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} checkout -q main && git -C {REPO_DIR} reset -q --hard origin/main
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements-colab.txt
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

import numpy as np, torch, ultralytics, porkqc
from pathlib import Path
from porkqc.config import load_yaml
env_info = {
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(), "torch": torch.__version__, "ultralytics": ultralytics.__version__,
    "git_commit": subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cpu": platform.processor() or platform.machine(), "cpu_threads": torch.get_num_threads(),
}
print(json.dumps(env_info, indent=2))
if not torch.cuda.is_available() and not QUICK:
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.")
from google.colab import drive
drive.mount("/content/drive")
cfg = load_yaml("configs/dataset.yaml")
DRIVE_DIR = Path(cfg["paths"]["drive_project_dir"])
WEIGHTS = DRIVE_DIR / "runs" / RUN_NAME / "weights" / "best.pt"
assert WEIGHTS.exists(), f"weights not found: {WEIGHTS}"
OUT = Path("results"); OUT.mkdir(exist_ok=True)
print("weights:", WEIGHTS)

## 3. Datasets (Roboflow download → time-blocked split)

In [ ]:
from porkqc import CLASS_NAMES
from porkqc.data.download import download_dataset, fix_data_yaml
from porkqc.data.resplit import collect_frames, assign_time_blocks, materialise_split
from porkqc.data.audit import list_images
from porkqc.models.detector import load_experiment, predict_images
from porkqc.predictions import save_json, load_json, export_ground_truth

rf = cfg["roboflow"]
data_root = Path(cfg["paths"]["colab_data_dir"])
if not (data_root / "data.yaml").exists():
    data_root = download_dataset(rf["workspace"], rf["project"], rf["version"], data_root, export_format=rf["export_format"])
fix_data_yaml(data_root / "data.yaml")
b = cfg["blocked_split"]
blocked = Path(b["colab_dir"])
frames_map, files_map, _ = collect_frames(data_root)
assign = assign_time_blocks(frames_map, b["block_size"], b["fractions"], b["gap"], b["seed"])
print("blocked split:", materialise_split(data_root, blocked, assign, CLASS_NAMES, files=files_map))
VALID = list_images(blocked / "valid" / "images")
TEST = list_images(blocked / "test" / "images")
exp = load_experiment(f"configs/experiments/{RUN_NAME.replace('_blocked', '')}.yaml")
PRED_KW = dict(exp["predict"])
print(len(VALID), "VALID,", len(TEST), "TEST images; predict settings", PRED_KW)

## 4. One-shot TEST evaluation (D-008, D-025, D-026)

Predictions on the blocked TEST split with exactly the settings used on VALID. The file is
saved to Drive immediately; if it already exists it is **reused, never regenerated**, so the
test set is used once.

In [ ]:
TEST_DRIVE = DRIVE_DIR / "test" / f"{RUN_NAME}_blocked_test.json"
TEST_LOCAL = OUT / "test" / TEST_DRIVE.name
TEST_LOCAL.parent.mkdir(parents=True, exist_ok=True)
if TEST_DRIVE.exists():
    print("TEST predictions already exist on Drive - reusing them (test set used once).")
else:
    pred = predict_images(WEIGHTS, TEST, RUN_NAME, "blocked_test", **PRED_KW)
    pred["git_commit"] = env_info["git_commit"]
    pred["decision_config_commit"] = subprocess.check_output(
        ["git", "log", "-1", "--format=%h %cI", "--", "configs/decision.yaml"], text=True).strip()
    TEST_DRIVE.parent.mkdir(parents=True, exist_ok=True)
    save_json(pred, TEST_DRIVE, indent=None)
    print("TEST predictions written:", TEST_DRIVE)
shutil.copy2(TEST_DRIVE, TEST_LOCAL)
save_json(export_ground_truth(blocked / "test", CLASS_NAMES, split="blocked_test"), OUT / "test" / "blocked_test_gt.json")
print("decision thresholds frozen at commit:", load_json(TEST_LOCAL).get("decision_config_commit"))

## 5. Accuracy vs input resolution and precision (VALID)

The model was trained at 640. Smaller inputs cut FLOPs roughly with pixel count; the question is
how much accuracy that costs — especially for the tiny packaging-error boxes. FP16 halves memory
traffic and uses tensor cores; does it change the predictions?

In [ ]:
PERF = OUT / "perf"; PERF.mkdir(exist_ok=True)
SIZES = [320, 480, 640, 800, 960] if not QUICK else [320, 640]
variants = [(s, 32) for s in SIZES] + ([(640, 16)] if torch.cuda.is_available() else [])
for size, bits in variants:
    out = PERF / f"valid_imgsz{size}_fp{bits}.json"
    if out.exists():
        continue
    kw = {**PRED_KW, "imgsz": size, **({"quantize": 16} if bits == 16 else {})}
    pred = predict_images(WEIGHTS, VALID, RUN_NAME, "blocked_valid", **kw)
    save_json(pred, out, indent=None)
    med = np.median([sum(e["speed_ms"].values()) for e in pred["images"]])
    print(f"imgsz {size} fp{bits}: median end-to-end {med:.2f} ms/image")

## 6. Latency study (pure forward pass, then end-to-end)

* **Forward pass only**: the fused network on a random tensor of the letterboxed shape
  (480 × 640 for 640 input), timed with device synchronisation, 20 warm-up + 100 timed runs.
* **Throughput** = batch / time. A production line sees one tray at a time, so **batch-1 latency**
  is what matters for the 250 ms budget; batching shows what the GPU *could* do.
* **CPU** (Colab's 2 vCPUs) answers: is a GPU needed at all?

In [ ]:
import pandas as pd
from ultralytics import YOLO
from porkqc.perf import time_callable, timing_summary

N_WARM, N_RUN = (20, 100) if not QUICK else (2, 5)
def letterbox_shape(size):          # Ultralytics pads 720x540 to a multiple of 32 keeping aspect
    h = int(np.ceil(size * 540 / 720 / 32) * 32)
    return h, size

def bench(device, batch, size, half):
    net = YOLO(str(WEIGHTS)).model.fuse().to(device).eval()
    net = net.half() if half else net.float()
    h, w = letterbox_shape(size)
    x = torch.rand(batch, 3, h, w, device=device, dtype=torch.float16 if half else torch.float32)
    sync = torch.cuda.synchronize if device == "cuda" else None
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    with torch.inference_mode():
        t = time_callable(lambda: net(x), N_WARM, N_RUN, sync)
    s = timing_summary(t)
    return {"device": device, "batch": batch, "imgsz": size, "input_hw": f"{h}x{w}", "precision": "fp16" if half else "fp32",
            **{f"ms_{k}": v for k, v in s.items() if k != "n"}, "n": s["n"],
            "ms_per_image": s["median"] / batch, "images_per_s": 1000 * batch / s["median"],
            "peak_mem_mb": torch.cuda.max_memory_allocated() / 1e6 if device == "cuda" else None}

rows = []
if torch.cuda.is_available():
    for bs in [1, 4, 8, 16, 32] if not QUICK else [1, 4]:
        for half in (False, True):
            rows.append(bench("cuda", bs, 640, half))
    for size in SIZES:
        rows.append(bench("cuda", 1, size, False))
cpu_runs = (5, 30) if not QUICK else (1, 3)
N_WARM_GPU, N_RUN_GPU = N_WARM, N_RUN
N_WARM, N_RUN = cpu_runs
for size in (320, 640):
    rows.append(bench("cpu", 1, size, False))
N_WARM, N_RUN = N_WARM_GPU, N_RUN_GPU
lat = pd.DataFrame(rows)
lat.to_csv(PERF / "latency.csv", index=False)
display(lat[["device", "batch", "imgsz", "precision", "ms_median", "ms_q25", "ms_q75", "ms_p95", "ms_per_image", "images_per_s", "peak_mem_mb"]].round(2))

### 6.1 Where does the time go? (profiler, batch 1, 640, FP32)

In [ ]:
from torch.profiler import profile, ProfilerActivity
dev = "cuda" if torch.cuda.is_available() else "cpu"
net = YOLO(str(WEIGHTS)).model.fuse().to(dev).eval().float()
x = torch.rand(1, 3, *letterbox_shape(640), device=dev)
acts = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if dev == "cuda" else [])
with torch.inference_mode():
    for _ in range(10):
        net(x)
    with profile(activities=acts) as prof:
        for _ in range(20):
            net(x)
        if dev == "cuda":
            torch.cuda.synchronize()
key = "cuda_time_total" if dev == "cuda" else "cpu_time_total"
table = prof.key_averages().table(sort_by=key, row_limit=15)
print(table)
(PERF / "profiler_top_ops.txt").write_text(table)

## 7. FLOPs and arithmetic intensity (roofline inputs)

In [ ]:
from ultralytics.utils.torch_utils import get_flops
from porkqc.perf import count_conv_costs, summarise_costs, machine_balance, T4_SPECS
net_cpu = YOLO(str(WEIGHTS)).model.fuse().float().eval()
bal32 = machine_balance(T4_SPECS["fp32_flops"], T4_SPECS["bandwidth"])
bal16 = machine_balance(T4_SPECS["fp16_flops"], T4_SPECS["bandwidth"])
flop_rows = []
for size in SIZES:
    for bs in ([1, 8, 32] if size == 640 else [1]):
        h, w = letterbox_shape(size)
        for nbytes, bal, prec in ((4, bal32, "fp32"), (2, bal16, "fp16")):
            s = summarise_costs(count_conv_costs(net_cpu, (bs, 3, h, w), nbytes), balance=bal)
            flop_rows.append({"imgsz": size, "batch": bs, "precision": prec, "input_hw": f"{h}x{w}", **s})
flops = pd.DataFrame(flop_rows)
flops.to_csv(PERF / "flops_intensity.csv", index=False)
display(flops.round(3))
check = {"ours_gflops_640x640": summarise_costs(count_conv_costs(net_cpu, (1, 3, 640, 640)))["gflops"],
         "ultralytics_gflops_640x640": float(get_flops(net_cpu, 640)),
         "machine_balance_fp32": bal32, "machine_balance_fp16": bal16}
print(check)
save_json(check, PERF / "flops_check.json")

## 8. Robustness: controlled perturbations of VALID

Same images and labels, one change at a time. Prediction settings identical to section 4.

In [ ]:
from PIL import Image
from porkqc.robustness import PERTURBATIONS, perturb
ROB = OUT / "robustness"; ROB.mkdir(exist_ok=True)
levels = PERTURBATIONS if not QUICK else {"exposure": [0.5], "blur": [2.0]}
for kind, vals in levels.items():
    for level in vals:
        out = ROB / f"valid_{kind}_{level}.json"
        if out.exists():
            continue
        d = Path(f"/content/perturbed/{kind}_{level}"); d.mkdir(parents=True, exist_ok=True)
        for k, p in enumerate(VALID):
            with Image.open(p) as im:
                Image.fromarray(perturb(np.asarray(im.convert("RGB")), kind, level, seed=k)).save(d / p.name, quality=95)
        pred = predict_images(WEIGHTS, sorted(d.iterdir()), RUN_NAME, f"blocked_valid_{kind}_{level}", **PRED_KW)
        pred["perturbation"] = {"kind": kind, "level": level}
        save_json(pred, out, indent=None)
        print("done", kind, level)
# one example image per perturbation, for the report
p0 = VALID[len(VALID) // 2]
with Image.open(p0) as im:
    base = np.asarray(im.convert("RGB"))
import matplotlib.pyplot as plt
items = [("clean", None)] + [(k, v) for k, vs in levels.items() for v in vs[:1] + vs[-1:]]
fig, axes = plt.subplots(1, len(items), figsize=(2.6 * len(items), 2.4))
for ax, (k, v) in zip(axes, items):
    ax.imshow(base if v is None else perturb(base, k, v)); ax.set_title(k if v is None else f"{k} {v}", fontsize=8); ax.axis("off")
fig.tight_layout(); fig.savefig(ROB / "examples.png", dpi=90); plt.show()

## 9. Baseline features (scikit-learn alternative, WP4)

324 hand-crafted features per image (`porkqc.models.baseline`) plus per-class presence labels.
The classifiers are trained afterwards on CPU.

In [ ]:
from porkqc.models.baseline import image_features, N_FEATURES
from porkqc.data.labels import read_label_file
from porkqc.data.resplit import scene_of
from porkqc.data.audit import frame_number
BASE = OUT / "baseline"; BASE.mkdir(exist_ok=True)
scene_by_file = scene_of(files_map)
for split in ("train", "valid", "test"):
    paths = list_images(blocked / split / "images")
    X = np.zeros((len(paths), N_FEATURES), np.float32)
    Y = np.zeros((len(paths), len(CLASS_NAMES)), np.int8)
    for i, p in enumerate(paths):
        X[i] = image_features(p)
        lab = read_label_file(blocked / split / "labels" / f"{p.stem}.txt", len(CLASS_NAMES))
        Y[i, np.unique(lab[:, 0].astype(int))] = 1
    np.savez_compressed(BASE / f"features_{split}.npz", X=X, Y=Y, files=np.array([p.name for p in paths]),
                        scene=np.array([scene_by_file.get(p.name, p.name) for p in paths]),
                        frame=np.array([frame_number(p) or -1 for p in paths]))
    print(split, X.shape, "defective share", Y.any(1).mean().round(3))

## 10. Save / push

In [ ]:
save_json(env_info, OUT / "notebook04_env.json")
shutil.copytree(OUT, DRIVE_DIR / "results_nb04", dirs_exist_ok=True)
from porkqc.env import get_secret
from porkqc.colab_utils import push_results
paths = [OUT / d for d in ("test", "perf", "robustness", "baseline")] + [OUT / "notebook04_env.json"]
try:
    if not PUSH_RESULTS:
        raise RuntimeError("PUSH_RESULTS is False")
    sha = push_results(REPO_DIR, [str(p) for p in paths], "results: notebook 04 (Colab)", get_secret("GITHUB_TOKEN"))
    print("Pushed to 'colab-results':", sha[:7])
except (RuntimeError, ValueError) as exc:
    print("Not pushed:", exc)
    archive = shutil.make_archive("/content/results_nb04", "zip", ".", "results")
    from google.colab import files
    files.download(archive)

Send Claude the zip (or say "pushed"), then *Runtime → Disconnect and delete runtime*.